# Removing all eddies with ice cover in the centre

In [2]:
import intake
import numpy as np
import matplotlib.pylab as plt
import xarray as xr
import matplotlib.colors as colors
import pandas as pd

In [3]:
#importing data

eerie_cat=intake.open_catalog("https://raw.githubusercontent.com/eerie-project/intake_catalogues/main/eerie.yaml")
da_ocean = eerie_cat["dkrz"]["disk"]["model-output"]["icon-esm-er"]["hist-1950"]["v20240618"]["ocean"]["gr025"]["2d_daily_mean"].to_dask()

In [4]:
da_ocean

<xarray.Dataset>
Dimensions:                              (time: 23741, lat: 721, lon: 1440,
                                          lev: 1, depth: 1)
Coordinates:
  * depth                                (depth) float64 1.0
  * lat                                  (lat) float64 -90.0 -89.75 ... 90.0
  * lev                                  (lev) float64 0.0
  * lon                                  (lon) float64 0.0 0.25 ... 359.5 359.8
  * time                                 (time) datetime64[ns] 1950-01-01T23:...
Data variables: (12/34)
    FrshFlux_IceSalt                     (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    FrshFlux_TotalIce                    (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    Wind_Speed_10m                       (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Evaporation    (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Precipitation  (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    atmos_fluxes_FrshFlux_Runoff         (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    ...                                   ...
    so                                   (time, depth, lat, lon) float32 dask.array<chunksize=(32, 1, 721, 1440), meta=np.ndarray>
    ssh                                  (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    stretch_c                            (time, lat, lon) float32 dask.array<chunksize=(32, 721, 1440), meta=np.ndarray>
    to                                   (time, depth, lat, lon) float32 dask.array<chunksize=(32, 1, 721, 1440), meta=np.ndarray>
    u                                    (time, depth, lat, lon) float32 dask.array<chunksize=(32, 1, 721, 1440), meta=np.ndarray>
    v                                    (time, depth, lat, lon) float32 dask.array<chunksize=(32, 1, 721, 1440), meta=np.ndarray>
Attributes: (12/31)
    Conventions:           CF-1.7 CMIP-6.2
    activity_id:           HighResMIP
    data_specs_version:    01.00.32
    forcing_index:         1
    initialization_index:  1
    license:               EERIE model data produced by MPI-M is licensed und...
    ...                    ...
    parent_activity_id:    HighResMIP
    sub_experiment_id:     none
    experiment:            coupled historical 1950-2014
    source:                ICON-ESM-ER (2023): \naerosol: none, prescribed MA...
    institution:           Max Planck Institute for Meteorology, Hamburg 2014...
    sub_experiment:        none

In [6]:
# da_ocean_chunk: copy of your dataset
da_ocean_chunk = da_ocean.isel(time=slice(0, 100))

In [11]:
# Compute the condition into memory
conc_mask = (da_ocean_chunk["conc"] == 1).compute()

# Apply the mask
da_ocean_chunk_ice = da_ocean_chunk.where(conc_mask, drop=True)

In [17]:
#importing modified eddy data

edso_ac = xr.open_dataset("/scratch/b/b383696/eddy_data/edso_ac.nc")
edso_c = xr.open_dataset("/scratch/b/b383696/eddy_data/edso_c.nc")

In [18]:
#creating new array for only lat,lon and time for eddies

lat_edso_ac  = xr.DataArray(edso_ac["latitude"].values,  dims="obs")
lon_edso_ac  = xr.DataArray(edso_ac["longitude"].values,  dims="obs")
time_edso_ac = xr.DataArray(edso_ac["time"].values, dims="obs")

lat_edso_c  = xr.DataArray(edso_c["latitude"].values,  dims="obs")
lon_edso_c  = xr.DataArray(edso_c["longitude"].values,  dims="obs")
time_edso_c = xr.DataArray(edso_c["time"].values, dims="obs")


In [19]:
#selecting ice concentration ("conc") over eddy centre

ice_edso_ac = da_ocean.conc.sel(lat = lat_edso_ac, lon = lon_edso_ac, time = time_edso_ac, method = "nearest")
ice_edso_c = da_ocean.conc.sel(lat = lat_edso_c, lon = lon_edso_c, time = time_edso_c, method = "nearest")

In [20]:
#deleting dimension "lev" -> needs same number of dimension as edso dataset

ice_edso_ac = ice_edso_ac.squeeze()
ice_edso_c = ice_edso_c.squeeze()

In [21]:
#changing conc_edso into a dataArray and adding it to the edso dataset

ice_edso_ac = xr.DataArray(ice_edso_ac.values, dims = "obs", name = "sea_ice")
ice_edso_ac.attrs["comment"] = "sea ice concentration over each eddy centre"
ice_edso_ac.attrs["long_name"] = "ice concentration"
edso_ac = edso_ac.assign({"sea_ice" : ice_edso_ac})

ice_edso_c = xr.DataArray(ice_edso_c.values, dims = "obs", name = "sea_ice")
ice_edso_c.attrs["comment"] = "sea ice concentration over each eddy centre"
ice_edso_c.attrs["long_name"] = "ice concentration"
edso_c = edso_c.assign({"sea_ice" : ice_edso_c})

In [22]:
edso_ac

<xarray.Dataset>
Dimensions:                        (obs: 2485451, NbSample: 50)
Dimensions without coordinates: obs, NbSample
Data variables: (12/30)
    amplitude                      (obs) float64 ...
    cost_association               (obs) float32 ...
    effective_area                 (obs) float32 ...
    effective_contour_height       (obs) float32 ...
    effective_contour_latitude     (obs, NbSample) float64 ...
    effective_contour_longitude    (obs, NbSample) float64 ...
    ...                             ...
    speed_radius                   (obs) float64 ...
    time                           (obs) datetime64[ns] 1993-01-01T23:59:59 ....
    track                          (obs) float64 ...
    uavg_profile                   (obs, NbSample) float64 ...
    clt                            (obs) float32 ...
    sea_ice                        (obs) float32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Anticyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T07:58:08Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [23]:
edso_c

<xarray.Dataset>
Dimensions:                        (obs: 2486794, NbSample: 50)
Dimensions without coordinates: obs, NbSample
Data variables: (12/30)
    amplitude                      (obs) float64 ...
    cost_association               (obs) float32 ...
    effective_area                 (obs) float32 ...
    effective_contour_height       (obs) float32 ...
    effective_contour_latitude     (obs, NbSample) float64 ...
    effective_contour_longitude    (obs, NbSample) float64 ...
    ...                             ...
    speed_radius                   (obs) float64 ...
    time                           (obs) datetime64[ns] 1993-01-01T23:59:59 ....
    track                          (obs) float64 ...
    uavg_profile                   (obs, NbSample) float64 ...
    clt                            (obs) float32 ...
    sea_ice                        (obs) float32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Cyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T08:03:27Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [24]:
#selecting only eddies without sea ice
edso_ac0 = edso_ac.where(edso_ac["sea_ice"] == 0, drop = True)
edso_c0 = edso_c.where(edso_c["sea_ice"] == 0, drop = True)

In [26]:
#adding ID to eddies
edso_ac0 = edso_ac0.assign(ID=("obs", np.arange(1, len(edso_ac0["obs"]) + 1)))
edso_c0 = edso_c0.assign(ID=("obs", np.arange(1, len(edso_c0["obs"]) + 1)))


In [27]:
edso_ac0

<xarray.Dataset>
Dimensions:                        (obs: 1986628, NbSample: 50)
Dimensions without coordinates: obs, NbSample
Data variables: (12/31)
    amplitude                      (obs) float64 0.008 0.01 ... 0.0324 0.0275
    cost_association               (obs) float32 0.2152 0.2833 ... 9.969e+36
    effective_area                 (obs) float32 5.428e+08 ... 5.388e+08
    effective_contour_height       (obs) float32 -0.048 -0.048 ... 0.188 0.196
    effective_contour_latitude     (obs, NbSample) float64 -75.94 ... -69.06
    effective_contour_longitude    (obs, NbSample) float64 203.5 203.6 ... 12.75
    ...                             ...
    time                           (obs) datetime64[ns] 1993-01-01T23:59:59 ....
    track                          (obs) float64 17.0 17.0 ... 5.648e+05
    uavg_profile                   (obs, NbSample) float64 0.0719 ... 0.1291
    clt                            (obs) float32 0.9306 0.9621 ... 0.6758 0.7295
    sea_ice                        (obs) float32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    ID                             (obs) int64 1 2 3 ... 1986626 1986627 1986628
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Anticyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T07:58:08Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [29]:
edso_c0

<xarray.Dataset>
Dimensions:                        (obs: 1949890, NbSample: 50)
Dimensions without coordinates: obs, NbSample
Data variables: (12/31)
    amplitude                      (obs) float64 -0.1169 -0.1162 ... -0.1393
    cost_association               (obs) float32 0.06921 0.1143 ... 9.969e+36
    effective_area                 (obs) float32 5.522e+09 ... 1.738e+10
    effective_contour_height       (obs) float32 0.076 0.068 ... -0.06 -0.06
    effective_contour_latitude     (obs, NbSample) float64 -70.13 ... -61.86
    effective_contour_longitude    (obs, NbSample) float64 355.8 355.5 ... 237.2
    ...                             ...
    time                           (obs) datetime64[ns] 1993-01-01T23:59:59 ....
    track                          (obs) float64 5.0 5.0 ... 6.283e+05 6.283e+05
    uavg_profile                   (obs, NbSample) float64 0.2521 ... 0.1695
    clt                            (obs) float32 0.9761 0.9765 ... 0.782 0.9581
    sea_ice                        (obs) float32 0.0 0.0 0.0 0.0 ... 0.0 0.0 0.0
    ID                             (obs) int64 1 2 3 ... 1949888 1949889 1949890
Attributes: (12/13)
    track_extra_variables:     lat_max,lon_max
    track_array_variables:     50
    array_variables:           contour_lat_e,contour_lon_e,contour_lat_s,cont...
    title:                     Cyclonic
    Metadata_Conventions:      Unidata Dataset Discovery v1.0
    comment:                   Surface product; mesoscale eddies
    ...                        ...
    framework_version:         0+unknown
    standard_name_vocabulary:  NetCDF Climate and Forecast (CF) Metadata Conv...
    date_created:              2025-09-05T08:03:27Z
    time_coverage_duration:    P8034D
    time_coverage_start:       1993-01-01T00:00:00Z
    time_coverage_end:         2014-12-31T00:00:00Z

In [32]:
#exporting ice-free centre eddies

edso_ac0.to_netcdf("/scratch/b/b383696/eddy_data/edso_ac0.nc")
edso_c0.to_netcdf("/scratch/b/b383696/eddy_data/edso_c0.nc")